In [1]:
!pip -q install -U anthropic datasets tqdm tenacity scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.2 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
print("Anthropic API key loaded securely.")

Enter your Anthropic API key: ··········
Anthropic API key loaded securely.


In [3]:
import os
import re
import json
import time
import pandas as pd

from json import JSONDecodeError
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset

from anthropic import (
    Anthropic,
    RateLimitError,
    APIConnectionError,
    BadRequestError,
    AuthenticationError
)

from tenacity import (
    retry,
    wait_exponential,
    stop_after_attempt,
    retry_if_exception_type
)

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

MODEL = "claude-opus-4-7"
DATASET_NAME = "ChanceFocus/flare-fsrl"

# First test with 100 samples.
# This dataset has fewer than 100 test samples, so it will evaluate all available samples.
LIMIT = 100

dataset = load_dataset(DATASET_NAME)

split_name = "test" if "test" in dataset else list(dataset.keys())[0]
split_data = dataset[split_name]

print(dataset)
print("Split:", split_name)
print("Columns:", split_data.column_names)
print("Example:", split_data[0])


def get_column_name(example, possible_names):
    for name in possible_names:
        if name in example:
            return name

    raise ValueError(
        f"Cannot find any column from {possible_names}. "
        f"Available columns are: {list(example.keys())}"
    )


example = split_data[0]

token_col = get_column_name(example, ["token", "tokens"])
label_col = get_column_name(example, ["label", "labels", "tags"])

print("Token column:", token_col)
print("Label column:", label_col)


def normalize_label(label):
    return str(label).strip()


# Build label set from dataset.
all_labels = set()

for row in split_data:
    for label in row[label_col]:
        all_labels.add(normalize_label(label))

LABELS = ["O"] + sorted([x for x in all_labels if x != "O"])

print("Number of labels:", len(LABELS))
print("Labels:", LABELS)

# Use short label codes to reduce output cost and parsing errors.
label_to_code = {label: f"L{i}" for i, label in enumerate(LABELS)}
code_to_label = {code: label for label, code in label_to_code.items()}

VALID_CODES = list(code_to_label.keys())


def build_label_mapping_text():
    lines = []
    for label in LABELS:
        code = label_to_code[label]
        lines.append(f"{code}: {label}")
    return "\n".join(lines)


LABEL_MAPPING_TEXT = build_label_mapping_text()

SYSTEM_PROMPT = """
You are a financial semantic role labeling model.

Task:
Given a list of tokens from a financial sentence, assign exactly one semantic role label code to each token.

The task is Textual Analogy Parsing / financial semantic role labeling.

Common role types include:
- QUANT: the measured quantity or financial metric
- VALUE: the numeric value
- TIME: time period
- THEME: the entity or object being described
- MANNER: direction or manner of change, such as rose, fell, increased, dropped
- LOCATION: location
- WHOLE: a larger group or whole object
- AGENT: actor or source
- SOURCE: information source

Rules:
1. Return exactly one label code for each input token.
2. The number of output labels must equal the number of input tokens.
3. Use only the allowed label codes.
4. Do not add explanations.
5. Return only valid JSON in this exact format:
{"labels": ["L0", "L1", "L0"]}
"""


def parse_claude_json(raw_text):
    raw_text = str(raw_text).strip()

    if raw_text == "":
        raise ValueError("Empty model output")

    try:
        return json.loads(raw_text)

    except JSONDecodeError:
        match = re.search(r"\{.*\}", raw_text, flags=re.DOTALL)

        if match:
            return json.loads(match.group(0))

        print("Raw model output:")
        print(repr(raw_text))
        raise


@retry(
    retry=retry_if_exception_type((RateLimitError, APIConnectionError, JSONDecodeError, ValueError)),
    wait=wait_exponential(multiplier=5, min=10, max=120),
    stop=stop_after_attempt(5),
    reraise=True
)
def call_claude_fsrl(tokens):
    indexed_tokens = "\n".join([f"{i}: {tok}" for i, tok in enumerate(tokens)])

    user_prompt = (
        f"Allowed label codes:\n{LABEL_MAPPING_TEXT}\n\n"
        f"Number of tokens: {len(tokens)}\n\n"
        f"Tokens:\n{indexed_tokens}\n\n"
        f"Return exactly {len(tokens)} label codes as JSON."
    )

    try:
        message = client.messages.create(
            model=MODEL,
            max_tokens=1500,
            system=SYSTEM_PROMPT,
            messages=[
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )

    except BadRequestError as e:
        print("\nBadRequestError detail:")
        print(e)
        raise

    except AuthenticationError as e:
        print("\nAuthenticationError detail:")
        print(e)
        raise

    raw_output = "".join(
        block.text for block in message.content
        if getattr(block, "type", None) == "text"
    )

    parsed = parse_claude_json(raw_output)

    if "labels" not in parsed:
        raise ValueError(f"Missing labels field in model output: {parsed}")

    return parsed


def fix_prediction_length(pred_codes, target_len):
    cleaned = []

    for code in pred_codes:
        code = str(code).strip()
        if code in code_to_label:
            cleaned.append(code)
        else:
            cleaned.append(label_to_code["O"])

    if len(cleaned) < target_len:
        cleaned = cleaned + [label_to_code["O"]] * (target_len - len(cleaned))

    if len(cleaned) > target_len:
        cleaned = cleaned[:target_len]

    return cleaned


def codes_to_labels(codes):
    return [code_to_label.get(code, "O") for code in codes]


def bio_to_spans(labels):
    spans = []
    start = None
    span_type = None

    for i, label in enumerate(labels + ["O"]):
        if label == "O":
            if start is not None:
                spans.append((start, i, span_type))
                start = None
                span_type = None
            continue

        if "-" not in label:
            continue

        prefix, label_type = label.split("-", 1)

        if prefix == "B":
            if start is not None:
                spans.append((start, i, span_type))
            start = i
            span_type = label_type

        elif prefix == "I":
            if start is None:
                start = i
                span_type = label_type
            elif span_type != label_type:
                spans.append((start, i, span_type))
                start = i
                span_type = label_type

    return spans


def compute_metrics(gold_all, pred_all):
    correct_tokens = 0
    total_tokens = 0
    exact_match_count = 0

    tp = 0
    fp = 0
    fn = 0

    for gold_labels, pred_labels in zip(gold_all, pred_all):
        total_tokens += len(gold_labels)
        correct_tokens += sum(g == p for g, p in zip(gold_labels, pred_labels))

        if gold_labels == pred_labels:
            exact_match_count += 1

        gold_spans = Counter(bio_to_spans(gold_labels))
        pred_spans = Counter(bio_to_spans(pred_labels))

        tp += sum((gold_spans & pred_spans).values())
        fp += sum((pred_spans - gold_spans).values())
        fn += sum((gold_spans - pred_spans).values())

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    token_accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0
    exact_match_accuracy = exact_match_count / len(gold_all) if len(gold_all) > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "token_accuracy": token_accuracy,
        "exact_match_accuracy": exact_match_accuracy,
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn
    }


sample_size = len(split_data) if LIMIT is None else min(LIMIT, len(split_data))
eval_data = split_data.select(range(sample_size))

rows = []
gold_all = []
pred_all = []

for idx, row in enumerate(tqdm(eval_data)):
    tokens = row[token_col]
    gold_labels = [normalize_label(x) for x in row[label_col]]

    try:
        result = call_claude_fsrl(tokens)
        pred_codes = result.get("labels", [])
        pred_codes = fix_prediction_length(pred_codes, len(gold_labels))
        pred_labels = codes_to_labels(pred_codes)
        error = ""

    except BadRequestError as e:
        print(f"\nStopping at row {idx} because of BadRequestError.")
        print(e)
        raise

    except AuthenticationError as e:
        print(f"\nStopping at row {idx} because of AuthenticationError.")
        print(e)
        raise

    except Exception as e:
        pred_labels = ["O"] * len(gold_labels)
        pred_codes = [label_to_code["O"]] * len(gold_labels)
        error = str(e)
        print(f"Error at row {idx}: {e}")

    gold_all.append(gold_labels)
    pred_all.append(pred_labels)

    rows.append({
        "idx": idx,
        "id": row.get("id", ""),
        "text": row.get("text", ""),
        "tokens": tokens,
        "gold_labels": gold_labels,
        "pred_codes": pred_codes,
        "pred_labels": pred_labels,
        "gold_spans": bio_to_spans(gold_labels),
        "pred_spans": bio_to_spans(pred_labels),
        "exact_match": gold_labels == pred_labels,
        "error": error
    })

    # Slow down to reduce rate-limit risk.
    time.sleep(5)


metrics = compute_metrics(gold_all, pred_all)

print("\n===== Claude Opus 4.7 FLARE-FSRL Evaluation =====")
print(f"Dataset: {DATASET_NAME}")
print(f"Split: {split_name}")
print(f"Model: {MODEL}")
print(f"Samples evaluated: {len(eval_data)}")
print(f"Precision / Correctness Rate: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"Exact Match Accuracy: {metrics['exact_match_accuracy']:.4f}")
print(f"Token Accuracy: {metrics['token_accuracy']:.4f}")
print(f"TP: {metrics['true_positive']}")
print(f"FP: {metrics['false_positive']}")
print(f"FN: {metrics['false_negative']}")

df = pd.DataFrame(rows)

df.to_csv("Claude_Opus_4_7_FLARE_FSRL_Evaluation_Results.csv", index=False)

with open("Claude_Opus_4_7_FLARE_FSRL_Metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

label_map = {
    "label_to_code": label_to_code,
    "code_to_label": code_to_label
}

with open("Claude_Opus_4_7_FLARE_FSRL_Label_Map.json", "w") as f:
    json.dump(label_map, f, indent=2)

num_errors = df["error"].astype(str).str.len().gt(0).sum()

print("\n===== Error Summary =====")
print(f"Failed samples: {num_errors}")
print(f"Total samples: {len(df)}")
print(f"Failure rate: {num_errors / len(df):.4f}")

pd.set_option("display.max_colwidth", 120)
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/519 [00:00<?, ?B/s]

data/test-00000-of-00001-cf2cb8d9b2882c9(…):   0%|          | 0.00/69.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/97 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['id', 'query', 'answer', 'text', 'label', 'token'],
        num_rows: 97
    })
})
Split: test
Columns: ['id', 'query', 'answer', 'text', 'label', 'token']
Example: {'id': 'fsrl0', 'query': "In the task of Textual Analogy Parsing (TAP), your job is to identify and label the semantic role of each token in a sentence. The labels can include 'O', 'I-QUANT', 'B-QUANT', 'I-TIME', 'B-TIME', 'I-MANNER', 'B-MANNER', 'I-THEME', 'B-THEME', 'I-VALUE', 'B-VALUE', 'I-WHOLE', 'B-WHOLE', 'I-LOCATION', 'B-LOCATION', 'I-AGENT', 'B-AGENT', 'I-CAUSE', 'B-CAUSE', 'I-SOURCE', 'B-SOURCE', 'I-REF_TIME', 'B-REF_TIME', 'I-CONDITION', 'B-CONDITION'. The 'B-' prefix indicates the beginning of a semantic role, while the 'I-' prefix denotes the continuation of that role. If a token doesn't correspond to any of these categories, assign it the 'O' label. Provide your answer as a sequence of 'token:label' pairs, with each pair on a new line.\nText: For the nine mon

100%|██████████| 97/97 [15:41<00:00,  9.71s/it]


===== Claude Opus 4.7 FLARE-FSRL Evaluation =====
Dataset: ChanceFocus/flare-fsrl
Split: test
Model: claude-opus-4-7
Samples evaluated: 97
Precision / Correctness Rate: 0.5027
Recall: 0.5336
F1 Score: 0.5177
Exact Match Accuracy: 0.0309
Token Accuracy: 0.6460
TP: 461
FP: 456
FN: 403

===== Error Summary =====
Failed samples: 0
Total samples: 97
Failure rate: 0.0000


,idx,id,text,tokens,gold_labels,pred_codes,pred_labels,gold_spans,pred_spans,exact_match,error
0,0,fsrl0,"For the nine months , earnings totaled $ 36.6 million , or $ 1.24 a share , down 46 % from $ 68.1 million , or $ 2.1...","[For, the, nine, months, ,, earnings, totaled, $, 36.6, million, ,, or, $, 1.24, a, share, ,, down, 46, %, from, $, ...","[O, B-TIME, I-TIME, I-TIME, O, B-QUANT, O, B-VALUE, I-VALUE, I-VALUE, O, O, B-VALUE, I-VALUE, I-VALUE, I-VALUE, O, O...","[L0, L9, L20, L20, L0, L8, L0, L10, L21, L21, L0, L0, L5, L16, L16, L16, L4, L21, L21, L0, L10, L21, L21, L0, L0, L5...","[O, B-TIME, I-TIME, I-TIME, O, B-THEME, O, B-VALUE, I-VALUE, I-VALUE, O, O, B-QUANT, I-QUANT, I-QUANT, I-QUANT, B-MA...","[(1, 4, TIME), (5, 6, QUANT), (7, 10, VALUE), (12, 16, VALUE), (21, 24, VALUE), (26, 30, VALUE), (31, 34, TIME)]","[(1, 4, TIME), (5, 6, THEME), (7, 10, VALUE), (12, 16, QUANT), (16, 17, MANNER), (17, 19, VALUE), (20, 23, VALUE), (...",False,
1,1,fsrl1,"For the first nine months of the year , Caterpillar said earnings fell 14 % to $ 390 million , or $ 3.85 a share , f...","[For, the, first, nine, months, of, the, year, ,, Caterpillar, said, earnings, fell, 14, %, to, $, 390, million, ,, ...","[O, B-TIME, I-TIME, I-TIME, I-TIME, I-TIME, I-TIME, I-TIME, O, B-THEME, O, B-QUANT, O, O, O, O, B-VALUE, I-VALUE, I-...","[L0, L9, L20, L20, L20, L20, L20, L20, L0, L1, L0, L8, L4, L10, L21, L0, L10, L21, L21, L0, L0, L10, L21, L16, L16, ...","[O, B-TIME, I-TIME, I-TIME, I-TIME, I-TIME, I-TIME, I-TIME, O, B-AGENT, O, B-THEME, B-MANNER, B-VALUE, I-VALUE, O, B...","[(1, 8, TIME), (9, 10, THEME), (11, 12, QUANT), (16, 19, VALUE), (21, 25, VALUE), (27, 30, VALUE), (32, 36, VALUE), ...","[(1, 8, TIME), (9, 10, AGENT), (11, 12, THEME), (12, 13, MANNER), (13, 15, VALUE), (16, 19, VALUE), (21, 23, VALUE),...",False,
2,2,fsrl2,"Merrill Lynch added 1 3\/4 to 28 , PaineWebber rose 3\/4 to 18 1\/2 and Bear Stearns rose 3\/8 to 14 1\/4 .","[Merrill, Lynch, added, 1, 3\/4, to, 28, ,, PaineWebber, rose, 3\/4, to, 18, 1\/2, and, Bear, Stearns, rose, 3\/8, t...","[B-THEME, I-THEME, B-MANNER, B-VALUE, I-VALUE, O, B-VALUE, O, B-THEME, B-MANNER, B-VALUE, O, B-VALUE, I-VALUE, O, B-...","[L8, L19, L4, L5, L16, L0, L10, L0, L8, L4, L5, L0, L10, L21, L0, L8, L19, L4, L5, L0, L10, L21, L0]","[B-THEME, I-THEME, B-MANNER, B-QUANT, I-QUANT, O, B-VALUE, O, B-THEME, B-MANNER, B-QUANT, O, B-VALUE, I-VALUE, O, B-...","[(0, 2, THEME), (2, 3, MANNER), (3, 5, VALUE), (6, 7, VALUE), (8, 9, THEME), (9, 10, MANNER), (10, 11, VALUE), (12, ...","[(0, 2, THEME), (2, 3, MANNER), (3, 5, QUANT), (6, 7, VALUE), (8, 9, THEME), (9, 10, MANNER), (10, 11, QUANT), (12, ...",False,
3,3,fsrl3,"At Ameritech , based in Chicago , customer access lines increased by 402,000 , or 2.6 % , and cellular mobile lines ...","[At, Ameritech, ,, based, in, Chicago, ,, customer, access, lines, increased, by, 402,000, ,, or, 2.6, %, ,, and, ce...","[O, B-THEME, O, O, O, B-LOCATION, O, B-QUANT, I-QUANT, I-QUANT, B-MANNER, O, B-VALUE, O, O, B-VALUE, I-VALUE, O, O, ...","[L0, L11, L0, L0, L0, L0, L0, L8, L19, L19, L4, L0, L10, L0, L0, L10, L21, L0, L0, L8, L19, L19, L4, L0, L10, L0, L0...","[O, B-WHOLE, O, O, O, O, O, B-THEME, I-THEME, I-THEME, B-MANNER, O, B-VALUE, O, O, B-VALUE, I-VALUE, O, O, B-THEME, ...","[(1, 2, THEME), (5, 6, LOCATION), (7, 10, QUANT), (10, 11, MANNER), (12, 13, VALUE), (15, 17, VALUE), (19, 22, QUANT...","[(1, 2, WHOLE), (7, 10, THEME), (10, 11, MANNER), (12, 13, VALUE), (15, 17, VALUE), (19, 22, THEME), (22, 23, MANNER...",False,
4,4,fsrl4,"The total short interest in Nasdaq stocks as of mid-October was 237.1 million shares , up from 223.7 million in Sept...","[The, total, short, interest, in, Nasdaq, stocks, as, of, mid-October, was, 237.1, million, shares, ,, up, from, 223...","[B-QUANT, I-QUANT, I-QUANT, I-QUANT, O, B-THEME, I-THEME, O, O, B-TIME, O, B-VALUE, I-VALUE, I-VALUE, O, O, O, B-VAL...","[L0, L0, L8, L19, L0, L11, L22, L0, L0, L9, L0, L10, L21, L16, L0, L4, L0, L10, L2